# 01 · 数据研究

**目的**:用管线的数据模块(`src/binance_data.py`)读取缓存,对最新数据做 exploration。

数据源(全部 Binance 公开 API,缓存在 `data/binance/{SYMBOL}.parquet`):
- 永续日 K:OHLCV + 成交额 + 笔数 + 主动买入量
- 资金费率:8h 结算按 UTC 日聚合(`funding_daily` = 当日多头实付)
- 现货收盘价/量:只为算基差 `(perp − spot)/spot`,无现货对的币为 NaN

In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent                      # launched from research/
sys.path.insert(0, str(ROOT / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import config
import binance_data as bd
import factors as F
import models as M

pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 40)
plt.rcParams['figure.figsize'] = (13, 5)
print(f"project root: {ROOT}")

## 1. 刷新数据(可选)

需要最新 bar 时把 `REFRESH` 改为 `True`(约 10 分钟,UTC 00:15 后跑才有昨日完整 bar)。

In [ ]:
REFRESH = False
if REFRESH:
    bd.download_universe()
    bd.collect_short_history()

## 2. 读取缓存 + 每币概览

In [ ]:
frames = bd.load_universe()

summary = pd.DataFrame([{
    'symbol': s,
    'bars': len(df),
    'start': df.index.min().date(),
    'end': df.index.max().date(),
    'avg_dollar_vol_30d(M)': df['QuoteVolume'].tail(30).mean() / 1e6,
    'has_spot': df['SpotClose'].notna().any(),
    'funding_mean_bps': df['funding_daily'].mean() * 1e4,
} for s, df in frames.items()]).sort_values('avg_dollar_vol_30d(M)', ascending=False)

print(f"{len(frames)} symbols | latest bar {summary['end'].max()}")
summary.reset_index(drop=True)

## 3. 数据质量检查

日历覆盖率(缺 bar)、关键列的 NaN 占比。管线原则:**从不填充**,NaN 表示"当时观测不到"。

In [ ]:
rows = []
for s, df in frames.items():
    full = pd.date_range(df.index.min(), df.index.max(), freq='D')
    rows.append({
        'symbol': s,
        'calendar_coverage': len(df) / len(full),
        'nan_close': df['Close'].isna().mean(),
        'nan_spot': df['SpotClose'].isna().mean(),
        'zero_volume_days': int((df['Volume'] == 0).sum()),
    })
quality = pd.DataFrame(rows).set_index('symbol')
display(quality.sort_values('calendar_coverage').head(10))
print("覆盖率<99% 的币:", quality[quality['calendar_coverage'] < 0.99].index.tolist() or '无')

## 4. 基准行情:BTC 价格与成交

In [ ]:
btc = frames[config.BENCHMARK_SYMBOL]
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 7), sharex=True,
                               gridspec_kw={'height_ratios': [3, 1]})
ax1.plot(btc.index, btc['Close'], lw=1.2)
ax1.set_ylabel('Close'); ax1.set_title(f'{config.BENCHMARK_SYMBOL} daily close')
ax1.grid(ls=':', alpha=0.5)
ax2.bar(btc.index, btc['QuoteVolume'] / 1e9, width=1)
ax2.set_ylabel('QuoteVol ($B)'); ax2.grid(ls=':', alpha=0.5)
plt.tight_layout(); plt.show()

## 5. 资金费率

多头在正资金费时付钱。全池日均资金费的时间序列 + 各币近 30 日均值排名——选币模型偏好负资金费(空头付钱)的币。

In [ ]:
fund = pd.DataFrame({s: df['funding_daily'] for s, df in frames.items()})
mkt_fund = fund.mean(axis=1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
ax1.plot(mkt_fund.index, mkt_fund.rolling(7).mean() * 1e4, lw=1)
ax1.axhline(0, color='k', lw=0.6)
ax1.set_title('全池日均资金费 (7d MA, bps)'); ax1.grid(ls=':', alpha=0.5)

recent = (fund.tail(30).mean() * 1e4).sort_values()
recent.head(10).plot.barh(ax=ax2, color='seagreen')
ax2.set_title('近 30 日资金费最低(做多收钱)的 10 个币 (bps/日)')
plt.tight_layout(); plt.show()
print(f"当前全池资金费为正的币占比: {(fund.tail(7).mean() > 0).mean():.0%}")

## 6. 基差(永续 − 现货)

In [ ]:
basis = pd.DataFrame({
    s: (df['Close'] - df['SpotClose']) / df['SpotClose']
    for s, df in frames.items() if df['SpotClose'].notna().any()})
mb = basis.mean(axis=1)
plt.plot(mb.index, mb.rolling(7).mean() * 1e4, lw=1)
plt.axhline(0, color='k', lw=0.6)
plt.title('全池平均基差 (7d MA, bps) — 正=升水(看多情绪)')
plt.grid(ls=':', alpha=0.5); plt.show()

## 7. 横截面结构:收益相关性与市场宽度

In [ ]:
rets = pd.DataFrame({s: np.log(df['Close']).diff() for s, df in frames.items()})
corr = rets.tail(180).corr()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5.5))
im = ax1.imshow(corr, cmap='RdYlBu_r', vmin=0, vmax=1)
ax1.set_title(f'近 180 日日收益相关性 (均值 {corr.values[np.triu_indices_from(corr, 1)].mean():.2f})')
ax1.set_xticks([]); ax1.set_yticks([])
plt.colorbar(im, ax=ax1, fraction=0.046)

breadth = (rets > 0).mean(axis=1)
ax2.plot(breadth.index, breadth.rolling(14).mean(), lw=1)
ax2.axhline(0.5, color='k', lw=0.6)
ax2.set_title('市场宽度:上涨币占比 (14d MA)'); ax2.grid(ls=':', alpha=0.5)
plt.tight_layout(); plt.show()

## 结论区(手写)

- 数据新鲜度 / 质量问题:
- 资金费与基差当前状态:
- 相关性水平对横截面策略容量的含义: